# Lesson 12 (Project 04): Runtime Feedback & Architectural Boundary Guards

In this project notebook, you will:
1. Diagnose a seeded large-file chunking bug using structured JSON runtime logs.
2. Run an AST-based architectural boundary checker (`check_architecture.py`).
3. Test Google Antigravity `PreToolUse` and `PostToolUse` hook contracts.

In [ ]:
!pip install -q -U google-genai

In [ ]:
import json

def pre_tool_use_guard(payload: dict) -> dict:
    cmd = payload.get("toolCall", {}).get("args", {}).get("CommandLine", "")
    if "rm -rf" in cmd or "DROP TABLE" in cmd.upper():
        return {"decision": "deny", "reason": "Blocked destructive command by harness safety gate."}
    return {"decision": "allow"}

blocked = pre_tool_use_guard({"toolCall": {"name": "run_command", "args": {"CommandLine": "rm -rf /"}}})
allowed = pre_tool_use_guard({"toolCall": {"name": "run_command", "args": {"CommandLine": "pytest -q"}}})
print("Blocked:", json.dumps(blocked))
print("Allowed:", json.dumps(allowed))
assert blocked["decision"] == "deny" and allowed["decision"] == "allow"